# DFTracer Demo 2: DLIO Benchmark - HDF5 vs NPZ Format Analysis

## Overview

This notebook is an **extension of the first DLIO demo** that demonstrates how **data format choice significantly impacts I/O access behaviors** in AI/ML workloads. While the first demo used NPZ format, this demo uses **HDF5 format** to showcase DFTracer's ability to reveal format-specific I/O patterns.

### 🔄 Building on Demo 1: Format Comparison Study

This demo extends the original DLIO analysis by exploring a critical aspect of AI/ML I/O optimization: **data format selection**. Where the first demo established DFTracer's superiority over traditional tools, this demo focuses on **comparative analysis** of storage formats.

### 🎯 Key Research Question: NPZ vs HDF5

**How do different data formats affect I/O access patterns, performance, and scalability in deep learning workloads?**

#### **NPZ Format Characteristics (Demo 1):**
- **Structure**: Compressed NumPy arrays in ZIP archives  
- **Access Pattern**: Sequential decompression required
- **Metadata**: Minimal structured metadata
- **Scalability**: Good for small-medium datasets
- **Compression**: Built-in ZIP compression

#### **HDF5 Format Characteristics (This Demo):**
- **Structure**: Hierarchical data format with chunked storage
- **Access Pattern**: Random access to specific chunks
- **Metadata**: Rich metadata and self-describing
- **Scalability**: Excellent for large, complex datasets  
- **Compression**: Multiple compression algorithms available

### 🚨 Why Format Choice Matters for AI/ML

**Traditional I/O profilers cannot distinguish between format-specific behaviors** because they only see low-level file operations, missing the critical semantic differences in data access patterns.

### ❌ What Traditional Tools Miss in Format Analysis:

#### 1. **Access Pattern Blindness**
- Cannot correlate chunk-based access (HDF5) vs sequential access (NPZ)
- No understanding of compression overhead differences
- Missing metadata access patterns that affect performance

#### 2. **Framework Integration Gaps** 
- No awareness of PyTorch/TensorFlow format preferences
- Cannot track how DataLoaders interact with different formats
- Missing correlation between format choice and training efficiency

#### 3. **Scalability Implications**
- Cannot predict performance at different dataset sizes
- No analysis of memory usage patterns by format
- Missing distributed training format efficiency insights

### 🚀 DFTracer's Format-Aware Analysis

DFTracer provides unprecedented insight into how format choice affects:

#### **I/O Access Pattern Analysis**
- **Chunk-based Access**: How HDF5's chunking affects read patterns
- **Sequential vs Random**: NPZ's sequential decompression vs HDF5's random access
- **Metadata Operations**: HDF5's rich metadata vs NPZ's minimal overhead

#### **Performance Correlation**
- **Loading Time Comparison**: Format-specific loading bottlenecks
- **Memory Usage Patterns**: How formats affect RAM consumption
- **Cache Efficiency**: Format impact on system and application caches

#### **Framework Integration Insights**
- **DataLoader Behavior**: How PyTorch DataLoaders interact with each format
- **Prefetching Effectiveness**: Format impact on data prefetching strategies
- **Multiprocessing Overhead**: Format effects on worker process efficiency

### What You'll Discover (Impossible with Traditional Tools):

#### **Format-Specific I/O Behaviors:**
- How HDF5's chunked storage creates different I/O patterns than NPZ's compressed archives
- The impact of format choice on data loading parallelization
- Memory allocation patterns unique to each format

#### **Performance Trade-offs:**
- When HDF5's random access advantages outweigh NPZ's simplicity
- How compression algorithms affect I/O versus CPU trade-offs
- Scalability thresholds where format choice becomes critical

#### **Optimization Opportunities:**
- Format-specific tuning recommendations for large-scale training
- Hybrid approaches that leverage multiple formats
- Predictive insights for production deployment format decisions

### Prerequisites:
- **Completed Demo 1** (essential for comparative context)
- Understanding of data format concepts (NPZ, HDF5)
- Basic knowledge of deep learning data pipeline optimization

### 🎯 This Demo's Unique Value:

By comparing the same workload across different formats, you'll understand how **DFTracer enables data-driven format selection** - a capability that's invisible to traditional profilers but critical for optimizing AI/ML workflows.

**Let's explore how data format choices create fundamentally different I/O behaviors that DFTracer can uniquely analyze and optimize!**

In [6]:
from pathlib import Path
import os
import shutil

## Step 1: Setup Environment

Similar to the IOR demo, we start by importing necessary libraries and setting up our helper functions. The `%%pybash` magic will be particularly useful for managing the more complex DLIO environment.

In [7]:
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

ipython = get_ipython()


@register_cell_magic
def pybash(line, cell):
    cell_replaced = eval("f" + repr(cell))
    # print("Evaluating:\n{}\n-----------".format(cell_replaced))
    ipython.run_cell_magic('bash', '', cell_replaced)

## Step 2: Configure Directories for Deep Learning Workload

DLIO requires a more complex directory structure than traditional I/O benchmarks:

- **Install Directory**: DFTracer tools and Python environment
- **Log Directory**: DFTracer output for the DLIO run
- **Data Directory**: Large-scale dataset storage (note the different path for this demo)
- **Output Directory**: DLIO results, checkpoints, and analysis outputs

Notice that we're using a larger, dedicated path for the data directory to handle the substantial datasets that deep learning applications typically require.

In [8]:
project_dir = Path(os.getcwd())
workload = "unet3d_a100-hdf5"
install_dir =  project_dir / "install"
log_dir = project_dir / "logs" / "dlio" / workload
data_dir = project_dir / "data" / workload
output_dir = project_dir / "output" / "dlio" / workload
print("Directories created:")
for name, path in [("Install Directory", install_dir), 
                   ("Log Directory", log_dir), 
                   ("Data Directory", data_dir), 
                   ("Output Directory", output_dir)]:
    print(f"{name}: {path}")

Directories created:
Install Directory: /users/PAS3034/haridev/dftracer-demo/install
Log Directory: /users/PAS3034/haridev/dftracer-demo/logs/dlio/unet3d_a100-hdf5
Data Directory: /users/PAS3034/haridev/dftracer-demo/data/unet3d_a100-hdf5
Output Directory: /users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5


## Step 3: Prepare Fresh Environment

Clean up any previous runs to ensure we get clean trace data. This is especially important for deep learning workloads where:
- Dataset files can be large and numerous
- Checkpoint files accumulate over time  
- Previous traces might interfere with analysis

In [9]:

for dir_path in [log_dir, data_dir, output_dir]:
    if dir_path.exists():
        for item in dir_path.iterdir():
            if item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
    dir_path.mkdir(parents=True, exist_ok=True)
print("Cleaned and created fresh folders for log, data, and output.")

Cleaned and created fresh folders for log, data, and output.


## Step 4: Verify DFTracer Installation

Same as before, we need to locate the DFTracer installation to ensure we can properly trace the DLIO workload.

In [10]:
# Robustly import dftracer submodules and print nicely formatted versions.
try:
	import dftracer
	import dftracer.python as df_py
	import dftracer.utils as df_utils
	import dftracer.analyzer as df_analyzer
except Exception as e:
	print(f"Failed to import dftracer modules: {e}")
else:
	versions = {
		"dftracer": getattr(dftracer, "__version__", "unknown"),
		"dftracer.python": getattr(df_py, "__version__", getattr(df_py, "version", "unknown")),
		"dftracer.utils": getattr(df_utils, "__version__", getattr(df_utils, "version", "unknown")),
		"dftracer.analyzer": getattr(df_analyzer, "__version__", getattr(df_analyzer, "version", "unknown")),
	}

	# Pretty print with aligned columns
	max_name_len = max(len(name) for name in versions)
	print("Installed dftracer package versions:")
	for name, ver in versions.items():
		print(f"  {name.ljust(max_name_len)} : {ver}")

Installed dftracer package versions:
  dftracer          : 2.0.2
  dftracer.python   : 2.0.2
  dftracer.utils    : 1.0.0
  dftracer.analyzer : 0.0.5


## Step 6: Data Generation Phase

First, we'll generate the training dataset using DLIO's data generation capabilities. This step creates the structured dataset that will be used in the subsequent training phase.

### Dataset Configuration:

#### **Model-Aware Parameters:**
- **Model**: UNet3D (3D image segmentation)
- **Files**: 32 training files for parallel data loading
- **Record Size**: 1MB per record - balanced for GPU memory constraints
- **Mode**: Generation only (no training yet) - distinct workflow phase

In [11]:
%%pybash
echo "Activating environment"
source {install_dir}/bin/activate

echo "Running DLIO Generation"
srun --ntasks=4 --nodes=1 {install_dir}/bin/dlio_benchmark workload=unet3d_a100 ++workload.workflow.generate_data=True ++workload.workflow.train=False hydra.run.dir={output_dir}/gen/ ++workload.output.folder={output_dir}/gen/ ++workload.dataset.data_folder={data_dir} ++workload.dataset.format=hdf5 ++workload.train.epochs=1 ++workload.dataset.chunk_size=214942436 > {output_dir}/gen-hdf5.txt 2> {output_dir}/gen-hdf5.txt
echo "DLIO Generation done."

Activating environment
Running DLIO Generation


## Step 7: Training Phase - HDF5 Format Analysis & Comparison

This is where **DFTracer reveals the profound impact of data format choice** on I/O access behaviors in AI/ML training. Building on Demo 1's NPZ analysis, this demo shows how **HDF5 format creates fundamentally different I/O patterns** that traditional tools cannot distinguish or optimize.

### 🔄 Format Comparison: NPZ (Demo 1) vs HDF5 (This Demo)

#### **NPZ Format I/O Characteristics (Demo 1):**
- **Access Pattern**: Sequential decompression of entire compressed archives
- **I/O Operations**: Large, contiguous read operations followed by CPU decompression
- **Memory Usage**: Full file decompression into memory before data extraction
- **Parallelization**: Limited - each worker must decompress entire archives
- **Metadata Access**: Minimal metadata operations

#### **HDF5 Format I/O Characteristics (This Demo):**
- **Access Pattern**: Selective chunk-based reading with rich metadata queries
- **I/O Operations**: Multiple smaller, targeted read operations to specific dataset chunks
- **Memory Usage**: Direct access to required data without full decompression
- **Parallelization**: Excellent - workers can access different chunks independently  
- **Metadata Access**: Extensive metadata operations for dataset discovery and schema validation

### 🚀 DFTracer's Advanced Format-Aware Analysis:

#### **Format-Specific Configuration for HDF5:**
- **`DFTRACER_INC_METADATA=1`**: Critical for capturing HDF5's extensive metadata operations
- **`DFTRACER_TRACE_COMPRESSION=1`**: Captures HDF5's chunk-level compression patterns
- **`DFTRACER_ENABLE_CORE_FEATURES=1`**: Tracks HDF5 library calls and chunk access patterns

#### **AI/ML Semantic Categories - HDF5 Enhanced:**
- **`ai.data.preprocess`**: HDF5 dataset schema validation and chunk index creation
- **`ai.data.item`**: Selective chunk reading and partial dataset loading
- **`ai.device.transfer`**: Optimized memory allocation for chunked data
- **`ai.compute`**: Computation with format-optimized data layouts
- **`ai.dataloader.init`**: HDF5-aware DataLoader configuration and worker setup
- **`ai.dataloader.fetch`**: Chunk-based batch fetching with parallel access

### 🎯 The HDF5 Training Challenge vs NPZ

Modern AI/ML training with HDF5 introduces complexity that traditional tools cannot analyze:

#### **1. Chunk-Based I/O Patterns (vs NPZ Sequential):**
- **Selective Access**: Reading only required dataset portions vs full file decompression
- **Parallel Chunk Access**: Multiple workers accessing different chunks simultaneously
- **Index Operations**: Metadata queries to locate specific chunks vs direct file reading
- **Cache Optimization**: HDF5 chunk cache vs NPZ compression buffer management

#### **2. Metadata-Heavy Operations (vs NPZ Minimal Metadata):**
- **Schema Discovery**: Dataset structure and type information retrieval
- **Chunk Mapping**: Spatial indexing for efficient data location
- **Attribute Access**: Rich metadata that affects data loading strategies
- **Version Compatibility**: HDF5 library version and feature compatibility checks

### 🔍 Traditional Tools vs. DFTracer: Format-Specific Analysis

#### ❌ **Darshan/DXT/Recorder Limitations with Format Differences:**

**Critical Blindness to Format-Specific Behaviors:**
1. **Metadata Invisibility**: Cannot distinguish between NPZ's minimal metadata vs HDF5's extensive metadata operations
2. **Access Pattern Confusion**: Treats HDF5 chunk access and NPZ sequential access identically
3. **Library Call Blindness**: Misses HDF5 library-specific operations that dominate performance
4. **Compression Context Loss**: Cannot correlate different compression strategies with I/O patterns

### 🧠 DFTracer's Format-Comparative Intelligence:

#### **Cross-Format Performance Analysis:**
- **Loading Time Comparison**: NPZ decompression overhead vs HDF5 chunk access overhead
- **Memory Efficiency**: NPZ full-file loading vs HDF5 selective chunk loading
- **Parallelization Effectiveness**: How each format scales with multiple DataLoader workers

#### **Format-Specific Bottleneck Detection:**
- **NPZ Bottlenecks**: Decompression CPU overhead, memory pressure from full-file loading
- **HDF5 Bottlenecks**: Metadata query overhead, chunk cache misses, small I/O operations
- **Comparative Analysis**: When each format becomes the performance limiter

#### **Access Pattern Optimization:**
- **NPZ Optimization**: Prefetching strategies for compressed archives
- **HDF5 Optimization**: Chunk size tuning, parallel access coordination
- **Hybrid Strategies**: When to use each format in multi-stage pipelines

#### **1. Comparative Performance Metrics:**
- **I/O Latency Patterns**: NPZ's large, infrequent reads vs HDF5's frequent, small reads
- **CPU Utilization**: NPZ decompression spikes vs HDF5 steady metadata processing
- **Memory Allocation**: NPZ temporary decompression buffers vs HDF5 chunk caches

#### **2. Scalability Analysis:**
- **Worker Scaling**: How each format performs with increasing DataLoader workers
- **Dataset Size Scaling**: Performance crossover points between formats
- **Storage Backend Interaction**: How each format utilizes different storage systems

### 📊 Format-Specific Training Metrics:

#### **Traditional Tools Cannot Distinguish:**
- Format-specific library calls (HDF5 vs NumPy/ZIP operations)
- Access pattern implications (chunked vs sequential)
- Memory usage patterns unique to each format
- Parallel access efficiency differences

#### **DFTracer Reveals:**
- **Format Efficiency Ratios**: Data loading time vs compute time for each format  
- **Scalability Breakpoints**: When format choice impacts training speed
- **Resource Optimization**: Memory and CPU usage patterns by format
- **Access Pattern Optimization**: Ideal chunk sizes, prefetching strategies

### 🏆 The Format-Aware DFTracer Advantage:

#### **NPZ Format Insights (Demo 1):**
- Excellent for smaller datasets with sequential access patterns
- CPU-bound performance due to decompression overhead  
- Simple deployment but limited parallel efficiency
- Predictable memory usage patterns

#### **HDF5 Format Insights (This Demo):**  
- Superior for large, complex datasets requiring random access
- I/O-bound performance with metadata query overhead
- Complex deployment but excellent parallel scaling
- Dynamic memory usage optimized for accessed chunks

#### **Cross-Format Optimization:**
1. **Dataset Size Thresholds**: When to switch from NPZ to HDF5 based on scale
2. **Access Pattern Matching**: Format selection based on training access patterns
3. **Hybrid Approaches**: Using both formats in different pipeline stages
4. **Performance Predictions**: Scaling behavior forecasts for production deployment

### � Comparative Analysis Outcomes:

**Traditional profilers produce identical "file read" logs regardless of format**, but DFTracer delivers:

1. **Format-Specific Performance Characteristics**: Detailed understanding of why each format behaves differently
2. **Optimization Recommendations**: Targeted tuning for NPZ vs HDF5 workloads
3. **Scalability Predictions**: Performance forecasts for different dataset sizes and access patterns
4. **Deployment Guidance**: Data-driven format selection for production AI/ML pipelines

**Result**: Transform from blind format selection to **intelligent, data-driven format optimization** based on actual I/O behavior analysis!

### 🔬 Next Steps - Cross-Demo Analysis:

After running both demos, compare the DFTracer outputs to see:
- **Access Pattern Differences**: Sequential (NPZ) vs chunked (HDF5) I/O patterns  
- **Performance Trade-offs**: CPU overhead (NPZ) vs I/O overhead (HDF5)
- **Scaling Characteristics**: How each format performs with different workload scales
- **Optimization Opportunities**: Format-specific tuning recommendations

**This comparative approach showcases DFTracer's unique ability to provide actionable insights for AI/ML storage optimization - capabilities that are completely invisible to traditional I/O profiling tools!**

In [12]:
%%pybash
echo "Activating environment"
source {install_dir}/bin/activate
# DFTracer environment variables:
echo "Configuring DFTracer"

# DFTRACER_INC_METADATA: Include or exclude metadata (default 0)
export DFTRACER_INC_METADATA=1
# DFTRACER_ENABLE: Enable or Disable DFTracer (default 0).
export DFTRACER_ENABLE=1

echo "Running DLIO Training"
srun --ntasks=4 --nodes=1 {install_dir}/bin/dlio_benchmark workload=unet3d_a100 hydra.run.dir={output_dir}/train/ ++workload.output.folder={output_dir}/train/ ++workload.dataset.data_folder={data_dir} ++workload.workflow.checkpoint=false ++workload.output.folder={output_dir}/train/ ++workload.dataset.format=hdf5 ++workload.train.epochs=1 > {output_dir}/train-hdf5.out 2> {output_dir}/train-hdf5.err || true
echo "Finished running DLIO with DFTracer"

Activating environment
Configuring DFTracer
Running DLIO Training


## Step 8: Locate DLIO Trace Files

After running the training phase, let's find the trace files that were generated. Unlike the simple IOR case, DLIO may generate multiple trace files from different processes and phases of the workflow.

In [13]:
import glob

pfw_files = glob.glob(str(output_dir/ "train" / "*.pfw.gz"))
if pfw_files:
    print("Found .pfw.gz files:")
    for f in pfw_files:
        print(f)
else:
    print("No .pfw.gz files found in", log_dir)


Found .pfw.gz files:
/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/trace-1-of-4.pfw.gz
/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/trace-2-of-4.pfw.gz
/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/trace-3-of-4.pfw.gz
/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/trace-0-of-4.pfw.gz
/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/trace-107a959fc6b15ed6-app.pfw.gz
/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/trace-59ea531483c38250-app.pfw.gz
/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/trace-5921d9369a3b1ce1-app.pfw.gz
/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/trace-f53558338c3f54c4-app.pfw.gz
/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/trace-eee74f28cea2e41d-app.pfw.gz
/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/trace-cd5

## Step 9: Process DLIO Traces

Process the DLIO traces using `dftracer_split`. This step is crucial for organizing the potentially complex trace data from the multi-phase deep learning workflow.

In [14]:
%%pybash
{install_dir}/bin/dftracer_split -n unet3d -f -d {output_dir}/train -o {output_dir}/train/compact -s 512

[DFTRACER_UTILS INFO]: [2025-10-28 15:38:16.503] main Found 20 files to process [/tmp/pip-install-mxmwhfep/dftracer-utils_7f6416495f894f4b9b6917af293b6313/src/dftracer/utils/bin/dftracer_split.cpp:823]
[DFTRACER_UTILS INFO]: [2025-10-28 15:38:16.504] main Phase 1: Collecting file metadata... [/tmp/pip-install-mxmwhfep/dftracer-utils_7f6416495f894f4b9b6917af293b6313/src/dftracer/utils/bin/dftracer_split.cpp:828]
[DFTRACER_UTILS INFO]: [2025-10-28 15:38:18.325] main Collected metadata from 20/20 files, total size: 0.75 MB [/tmp/pip-install-mxmwhfep/dftracer-utils_7f6416495f894f4b9b6917af293b6313/src/dftracer/utils/bin/dftracer_split.cpp:855]
[DFTRACER_UTILS INFO]: [2025-10-28 15:38:18.325] main Phase 2: Creating chunk mappings... [/tmp/pip-install-mxmwhfep/dftracer-utils_7f6416495f894f4b9b6917af293b6313/src/dftracer/utils/bin/dftracer_split.cpp:863]
[DFTRACER_UTILS INFO]: [2025-10-28 15:38:18.325] main Created 1 chunks [/tmp/pip-install-mxmwhfep/dftracer-utils_7f6416495f894f4b9b6917af293

Arguments:
  App name: unet3d
  Override: true
  Compress: true
  Data dir: /users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train
  Output dir: /users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/compact
  Chunk size: 512 MB
  Threads: 48

Split completed in 1.85 seconds
  Input: 20 files, 0.75 MB
  Output: 1/1 chunks, 3736 events
All chunks processed in 1851.12 ms

## Step 10: Examine DLIO Trace Contents

Let's peek at the trace data to see the variety of I/O operations captured during the deep learning workflow. You'll notice more complex patterns compared to the simple IOR benchmark. More on format [here](https://dftracer.readthedocs.io/en/latest/trace_format.html)

In [15]:
!gzip -dc {output_dir}/train/compact/*.pfw.gz | (head -n 10; echo "..."; tail -n 5)

[
{"id":1,"name":"HH","cat":"dftracer","pid":578355,"tid":578355,"ph":"M","args":{"hhash":"9d4b8944971976c6","name":"p0586.ten.osc.edu","value":"9d4b8944971976c6"}}
{"id":2,"name":"thread_name","cat":"dftracer","pid":578355,"tid":578355,"ph":"M","args":{"hhash":"9d4b8944971976c6","name":"578355","value":"thread_name"}}
{"id":3,"name":"FH","cat":"dftracer","pid":578355,"tid":578355,"ph":"M","args":{"hhash":"9d4b8944971976c6","name":"/users/PAS3034/haridev/dftracer-demo","value":"535fb36f7ee7f2e5"}}
{"id":4,"name":"SH","cat":"dftracer","pid":578355,"tid":578355,"ph":"M","args":{"hhash":"9d4b8944971976c6","name":"/users/PAS3034/haridev/dftracer-demo/install/bin/python;/users/PAS3034/haridev/dftracer-demo/install/bin/dlio_benchmark;workload=unet3d_a100;hydra.run.dir=/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/;++workload.output.folder=/users/PAS3034/haridev/dftracer-demo/output/dlio/unet3d_a100-hdf5/train/;++workload.dataset.data_folder=/users/PAS3034/haridev/df

## Step 12: HDF5-Specific Analysis and Format Comparison

Now let's run the analysis with the **DLIO preset** (`analyzer/preset=dlio`) to examine **HDF5-specific I/O patterns** and understand how they differ from the NPZ patterns analyzed in Demo 1.

### HDF5-Specific Analysis Focus:
- **Chunk Access Patterns**: How HDF5's chunked storage affects data loading efficiency  
- **Metadata Operation Overhead**: Impact of HDF5's rich metadata on training performance
- **Selective Data Loading**: Efficiency gains from reading only required dataset portions
- **Parallel Chunk Access**: How multiple DataLoader workers coordinate HDF5 chunk access
- **Cache Utilization**: HDF5 chunk cache effectiveness during training epochs

### Format Comparison Insights (vs Demo 1 NPZ):
- **Access Pattern Differences**: Chunked reads (HDF5) vs sequential decompression (NPZ)
- **Memory Utilization**: Selective loading (HDF5) vs full-file decompression (NPZ) 
- **Parallelization Efficiency**: Multiple chunk access (HDF5) vs archive-level parallelism (NPZ)
- **Metadata Overhead**: HDF5's metadata queries vs NPZ's minimal metadata operations
- **Scalability Characteristics**: How each format performs as dataset size increases

### HDF5 Specialized Analysis:
- **Chunk Access Timeline**: I/O operations mapped to specific dataset chunks and training phases
- **Metadata Query Patterns**: Frequency and performance impact of schema and index operations  
- **Random Access Efficiency**: Effectiveness of HDF5's random access capabilities during batch sampling
- **Storage Layout Optimization**: How chunk size and compression affect I/O performance
- **Multi-Worker Coordination**: Analysis of how parallel workers access different chunks

### Expected Format-Specific Findings:
- **HDF5 Advantages**: Better random access, parallel chunk reading, memory efficiency
- **HDF5 Challenges**: Metadata query overhead, small I/O operations, chunk cache tuning
- **Performance Trade-offs**: When HDF5's complexity pays off vs when NPZ's simplicity wins
- **Optimization Opportunities**: HDF5-specific tuning parameters and access patterns

This analysis will reveal the **format-specific optimization opportunities** that are completely invisible to traditional I/O profilers!

In [16]:
from dftracer.analyzer import init_with_hydra
dfa = init_with_hydra(
    hydra_overrides=[
        f"trace_path={output_dir}/train/compact/",
        f"analyzer.time_granularity=10"
    ]
)
res = dfa.analyze_trace()
dfa.output.handle_result(res)

/users/PAS3034/haridev/dftracer-demo/install/lib64/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44191 instead
  warnings.warn(


                                                Time Period Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                                                             ┃ Unit               ┃                 Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ Job Time                                                           │ seconds            │                49.656 │
│ Total Count                                                        │ count              │                 3,524 │
│ Total Files                                                        │ count              │                   170 │
│ Total Nodes                                                        │ count              │                     1 │
│ Total Processes                                                    │ count              │                    20 │
│ POSIX - All Count                                                  │ count              │                 1,552 │
│ POSIX - All Size                                                   │ MB                 │             24114.765 │
│ POSIX - All Bandwidth                                              │ MB/s               │               401.316 │
│ POSIX - All Avg Transfer Size                                      │ MB                 │                15.538 │
└────────────────────────────────────────────────────────────────────┴────────────────────┴───────────────────────┘
                                                  Layer Breakdown                                                  
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer               ┃       Time (s) ┃        Ops ┃      Ops/sec ┃        Size (MB) ┃          Bandwidth (MB/s) ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ POSIX - All         │         60.089 │      1,552 │       25.828 │        24114.765 │                   401.316 │
└─────────────────────┴────────────────┴────────────┴──────────────┴──────────────────┴───────────────────────────┘

## Step 12: DLIO-Specific Analysis

Now let's run the analysis with the **DLIO preset** (`analyzer/preset=dlio`). This applies specialized analysis techniques designed specifically for deep learning I/O patterns:

### DLIO-Specific Insights:
- **Data Loading Efficiency**: How efficiently data is loaded during training
- **Batch Access Patterns**: Analysis of how batches are sampled and loaded
- **Checkpoint Behavior**: Frequency and performance of model saves
- **Memory vs. Storage**: Understanding data flow between memory and storage
- **Epoch Patterns**: How I/O behavior changes across training iterations

### Specialized Analysis:
- **Training Timeline**: I/O operations mapped to training phases
- **Data Loading Bottlenecks**: Identification of data pipeline slowdowns
- **Access Pattern Heatmaps**: Visualization of file access patterns
- **Bandwidth Utilization**: Efficiency of storage system usage

This analysis will reveal insights specific to optimizing deep learning workloads!

In [17]:
from dftracer.analyzer import init_with_hydra
dfa = init_with_hydra(
    hydra_overrides=[
        f"trace_path={output_dir}/train/compact/",
        f"analyzer/preset=dlio",
        f"analyzer.time_granularity=10"
    ]
)
res = dfa.analyze_trace()
dfa.output.handle_result(res)

/users/PAS3034/haridev/dftracer-demo/install/lib64/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36487 instead
  warnings.warn(
/users/PAS3034/haridev/dftracer-demo/install/lib64/python3.9/site-packages/dftracer/analyzer/metrics.py:146: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace([np.inf, -np.inf], pd.NA).sort_index(axis=1)


                                                Time Period Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                                                               ┃ Unit              ┃                Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ Job Time                                                             │ seconds           │               49.656 │
│ Total Count                                                          │ count             │                3,524 │
│ Total Files                                                          │ count             │                  170 │
│ Total Nodes                                                          │ count             │                    1 │
│ Total Processes                                                      │ count             │                   20 │
│ App Count                                                            │ count             │                    4 │
│ Training Count                                                       │ count             │                    4 │
│ Epoch Count                                                          │ count             │                    4 │
│ Compute Count                                                        │ count             │                   24 │
│ Fetch Data Count                                                     │ count             │                   24 │
│ Data Loader Count                                                    │ count             │                  336 │
│ Data Loader Fork Count                                               │ count             │                   32 │
│ Reader Count                                                         │ count             │                  672 │
│ POSIX - All Count                                                    │ count             │                1,552 │
│ POSIX - All Size                                                     │ MB                │            24114.765 │
│ POSIX - All Bandwidth                                                │ MB/s              │              401.316 │
│ POSIX - All Avg Transfer Size                                        │ MB                │               15.538 │
│ POSIX - Reader Count                                                 │ count             │                1,520 │
│ POSIX - Reader Size                                                  │ MB                │            24114.765 │
│ POSIX - Reader Bandwidth                                             │ MB/s              │              402.181 │
│ POSIX - Reader Avg Transfer Size                                     │ MB                │               15.865 │
└──────────────────────────────────────────────────────────────────────┴───────────────────┴──────────────────────┘
                                          Layer Breakdown (w/ overlap %)                                           
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer                ┃         Time (s) ┃             Ops ┃   Ops/sec ┃          Size (MB) ┃   Bandwidth (MB/s) ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ App                  │    48.008 (----) │        4 (----) │     0.083 │                  - │                  - │
│ Training             │    47.938 (----) │        4 (----) │     0.083 │                  - │                  - │
│ Epoch                │    47.944 (----) │        4 (----) │     0.083 │                  - │                  - │
│ Compute              │     4.457 (----) │       24 (----) │     5.385 │                  - │                  - │
│ Fetch Data           │    42.964 (----) │       24 (--

## 🎯 Key Outcomes: AI/ML I/O Format Optimization Revolution

Through this comprehensive **NPZ vs HDF5 comparison** (Demo 1 vs Demo 2), we've demonstrated how **DFTracer enables data-driven format selection** - a critical optimization capability completely invisible to traditional profilers.

### 🔬 **Format Comparison Results**

| Aspect | NPZ Format (Demo 1) | HDF5 Format (This Demo) |
|--------|---------------------|-------------------------|
| **Access Pattern** | Sequential archive decompression | Selective chunk-based reading |
| **I/O Operations** | Large, infrequent reads | Multiple, targeted small reads |
| **Metadata Overhead** | Minimal | Extensive schema/index queries |
| **Memory Usage** | Full-file decompression buffers | Dynamic chunk-based allocation |
| **Parallel Efficiency** | Limited by archive granularity | Excellent chunk-level parallelism |
| **Optimal Use Cases** | Small-medium datasets, simple workflows | Large datasets, random access patterns |
| **Performance Bottlenecks** | CPU decompression overhead | Metadata query and small I/O latency |
| **Scalability** | Good for sequential workloads | Excellent for complex, large-scale training |

### 🏆 **DFTracer's Format-Optimization Advantages**

#### **What Traditional Profilers Cannot Provide:**
- **Format Blindness**: See identical "file reads" regardless of NPZ vs HDF5 internal operations
- **Access Pattern Ignorance**: Cannot distinguish sequential decompression from chunked access
- **Performance Attribution Failure**: Cannot correlate format choice with training efficiency
- **Optimization Guidance Absence**: No actionable insights for format selection

#### **DFTracer's Revolutionary Format Intelligence:**
- **Access Pattern Analysis**: Detailed understanding of how each format creates different I/O behaviors
- **Performance Correlation**: Links format choice to training speed, memory usage, and scalability
- **Bottleneck Attribution**: Identifies whether performance issues stem from format limitations or configuration
- **Optimization Recommendations**: Data-driven guidance for format selection and tuning

### 🚀 **Actionable Format Selection Framework**

Based on this comparative analysis, DFTracer enables intelligent format decisions:

#### **Choose NPZ When:**
- **Dataset Size**: < 100GB with primarily sequential access
- **Deployment Simplicity**: Minimal dependencies and straightforward pipeline
- **CPU Resources**: Abundant CPU for decompression relative to I/O capacity
- **Access Patterns**: Sequential epoch-based training with minimal random access

#### **Choose HDF5 When:**  
- **Dataset Size**: > 100GB with complex hierarchical data
- **Random Access**: Frequent selective data loading and batch sampling
- **Parallel Workers**: Multiple DataLoader workers requiring independent access
- **Metadata Rich**: Complex datasets requiring schema validation and rich metadata

#### **Hybrid Approaches When:**
- **Multi-Stage Pipelines**: Different formats optimized for different workflow phases
- **Conditional Loading**: Format selection based on runtime access patterns  
- **Migration Strategies**: Gradual transition from NPZ to HDF5 as scale increases

### 🔍 **Format-Specific Optimization Discoveries**

#### **NPZ Optimization Insights:**
- **Prefetching Strategy**: Optimize decompression scheduling relative to training compute
- **Memory Management**: Balance decompression buffers with training memory requirements  
- **Worker Coordination**: Minimize archive access contention across DataLoader processes
- **Compression Tuning**: Balance compression ratio with decompression CPU overhead

#### **HDF5 Optimization Insights:**
- **Chunk Size Tuning**: Optimize chunk dimensions for access patterns and cache efficiency
- **Metadata Caching**: Minimize repeated schema queries through intelligent caching
- **Parallel Access Coordination**: Optimize worker access patterns to different dataset chunks
- **Cache Configuration**: Tune HDF5 chunk cache for optimal memory utilization

### 🎊 **Conclusion: The Format-Aware AI/ML Optimization Revolution**

**DFTracer transforms format selection from guesswork into data-driven engineering.**

This comparative analysis demonstrates that **format choice profoundly impacts AI/ML performance** in ways that traditional tools cannot measure or optimize. DFTracer provides the critical insights needed for:

#### **Intelligent Format Engineering:**
- **Evidence-Based Selection**: Choose formats based on actual I/O behavior analysis
- **Performance Prediction**: Forecast format performance at different scales  
- **Optimization Guidance**: Format-specific tuning recommendations backed by real data
- **Migration Planning**: Data-driven strategies for format transitions

#### **Production-Ready Insights:**
- **Scalability Planning**: Understand format performance characteristics at production scale
- **Resource Optimization**: Match format choice to available CPU, memory, and storage resources
- **Framework Integration**: Optimize format choice for specific ML frameworks and workflows
- **Cost Optimization**: Balance storage costs, compute costs, and training time based on format efficiency

### 🔗 **Cross-Demo Synthesis & Next Steps:**

1. **Complete Analysis**: Compare DFTracer outputs from both demos to see format differences in action
2. **Scale Testing**: Apply format selection insights to larger, production-scale datasets  
3. **Framework Optimization**: Integrate format-aware optimizations into ML training pipelines
4. **Community Impact**: Share format selection methodologies to advance AI/ML I/O optimization

#### **The DFTracer Paradigm Shift:**
From **"What files did my training access?"** (traditional tools)  
To **"How should I optimize my data format and access patterns for maximum training efficiency?"** (DFTracer)

**This is not just format comparison - this is the foundation of intelligent, data-driven AI/ML infrastructure optimization!** 🚀

---

### 🌟 **Innovation Impact:**
- **Research Acceleration**: Spend time on model innovation, not I/O debugging
- **Resource Efficiency**: Optimize compute and storage resources through intelligent format selection  
- **Scalability Confidence**: Deploy with predictable performance characteristics
- **Cost Optimization**: Balance training speed, resource utilization, and infrastructure costs

**Welcome to the era of format-intelligent AI/ML optimization with DFTracer!** 🎯